# 53 — Long-generation composition (greedy tier)

Tests whether blend, normalized score voting, and round-robin scheduling
retain their two-concept effects over the 200-step horizon selected in
`long-generation-inputs.yaml`. Each method is run at lambda 0 and 0.4.

The notebook is safe by default: it resolves and prints the complete run
matrix without loading model weights. Set `RUN_GENERATION = True` for a
smoke run or provide the printed confirmation string for the full matrix.
Existing JSONL logs are resumed by cell identity unless `OVERWRITE = True`.


In [1]:
# Papermill parameters
RUN_GENERATION = False
SMOKE = False
SMOKE_STEPS = 8
OVERWRITE = False
CONFIRM_FULL_RUN = ""
RUN_BENCHMARK = False
BENCHMARK_ONLY = False
RUN_TRAJECTORIES = False


In [2]:
# Parameters
RUN_GENERATION = True
SMOKE = False
OVERWRITE = False
CONFIRM_FULL_RUN = "RUN_53_450_CELLS"
RUN_BENCHMARK = False
BENCHMARK_ONLY = False
RUN_TRAJECTORIES = True


In [3]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from frames.evaluation.harness import EvaluationHarness
from frames.experiments.long_generation import (
    generation_metadata,
    load_long_generation_experiment,
    token_bins,
)
from frames.experiments.notebook_helpers import llama_chat
from frames.linalg.angles import vector_subspace_cosine
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

CONFIG_PATH = Path("long-generation-inputs.yaml")
EXPERIMENT = load_long_generation_experiment(CONFIG_PATH, "experiment_53")

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
METHODS = ("blend", "vote", "schedule")
BIN_WIDTH = 20

prompts_by_id = {
    row["id"]: llama_chat(row["user"], row["assistant_prefix"])
    for row in EXPERIMENT.prompts
}
active_prompts = list(EXPERIMENT.prompts)
active_pairs = list(EXPERIMENT.composition_pairs)
active_lambdas = list(EXPERIMENT.lambdas)
run_steps = EXPERIMENT.steps
if SMOKE:
    active_prompts = active_prompts[:1]
    active_pairs = active_pairs[:1]
    run_steps = min(SMOKE_STEPS, EXPERIMENT.steps)

full_cells = EXPERIMENT.guided_run_count(len(METHODS))
active_cells = (
    len(active_prompts) * len(active_pairs) * len(METHODS) * len(active_lambdas)
)
confirmation = f"RUN_53_{full_cells}_CELLS"
print(
    f"full matrix: {len(EXPERIMENT.composition_pairs)} pairs x "
    f"{len(EXPERIMENT.prompts)} prompts x {len(METHODS)} methods x "
    f"{len(EXPERIMENT.lambdas)} lambdas = {full_cells} guided continuations"
)
print(
    f"nominal full token budget: "
    f"{EXPERIMENT.output_token_budget(len(METHODS)):,} (early EOS excluded)"
)
print(
    f"this execution: {active_cells} cells at steps={run_steps}; "
    f"SMOKE={SMOKE}, RUN_GENERATION={RUN_GENERATION}"
)
print(f"full-run confirmation: {confirmation}")

if (
    RUN_GENERATION
    and not SMOKE
    and not BENCHMARK_ONLY
    and CONFIRM_FULL_RUN != confirmation
):
    raise RuntimeError(
        "Full generation is locked. Set CONFIRM_FULL_RUN to the printed string."
    )
if BENCHMARK_ONLY and not RUN_BENCHMARK:
    raise ValueError("BENCHMARK_ONLY requires RUN_BENCHMARK=True")
if SMOKE_STEPS < 1:
    raise ValueError("SMOKE_STEPS must be positive")


full matrix: 15 pairs x 5 prompts x 3 methods x 2 lambdas = 450 guided continuations
nominal full token budget: 90,000 (early EOS excluded)
this execution: 450 cells at steps=200; SMOKE=False, RUN_GENERATION=True
full-run confirmation: RUN_53_450_CELLS


In [4]:
suffix = "_smoke" if SMOKE else ""
LOG_PATH = Path(f"resources/53_long_composition{suffix}.jsonl")
BASELINE_PATH = Path(f"resources/53_long_composition_baselines{suffix}.json")
SUMMARY_PATH = Path(f"resources/53_long_composition_summary{suffix}.csv")
TRAJECTORY_PATH = Path(
    f"resources/53_long_composition_trajectories{suffix}.csv"
)
TRAJECTORY_FIGURE_PATH = Path(
    f"resources/53_long_composition_trajectories{suffix}.png"
)
FIGURE_PATH = Path(f"resources/53_long_composition_metrics{suffix}.png")
TIMING_PATH = Path("resources/53_long_composition_timing.csv")

def read_records(path):
    if not path.exists() or not path.read_text().strip():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line]


def record_key(config):
    return (
        config["pair_id"],
        config["prompt_id"],
        config["method"],
        float(config["lam"]),
        int(config["steps"]),
    )


if RUN_GENERATION and OVERWRITE:
    for path in (LOG_PATH, BASELINE_PATH, SUMMARY_PATH, TRAJECTORY_PATH):
        path.unlink(missing_ok=True)

planned = {
    (pair["id"], prompt["id"], method, float(lam), run_steps)
    for pair in active_pairs
    for prompt in active_prompts
    for method in METHODS
    for lam in active_lambdas
}
completed = {record_key(row["config"]) for row in read_records(LOG_PATH)}
unexpected = completed - planned
if unexpected:
    raise RuntimeError(
        f"{LOG_PATH} contains {len(unexpected)} cells outside the resolved "
        "matrix; preserve it and start a new artifact or use OVERWRITE=True"
    )
print(f"resume state: {len(completed)}/{active_cells} active cells already logged")


resume state: 0/450 active cells already logged


## Model, concepts, and shared baselines

The model is loaded only for generation or the optional longitudinal
analysis. Baselines are versioned by model, prompt text, and run horizon.
A stale baseline cache is rejected instead of silently reused.


In [5]:
need_model = RUN_GENERATION or RUN_TRAJECTORIES
fur = harness = None
concepts = {}
baseline_texts_by_id = {}

if need_model:
    fur = FrameUnembeddingRepresentation.from_model_id(
        id=MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16
    )
    harness = EvaluationHarness(
        fur,
        LOG_PATH,
        min_lemmas_per_synset=MIN_LEMMAS,
        max_token_count=MAX_TOKENS,
    )

    used_synsets = sorted(
        {name for pair in active_pairs for name in (pair["a"], pair["b"])}
    )
    for name in used_synsets:
        cached = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
        concepts[name] = Concept(
            synset=cached.synset, tensor=cached.tensor.to(fur.model.device)
        )
    print(f"loaded {len(concepts)} cached concepts")

    need_baselines = (RUN_GENERATION and not BENCHMARK_ONLY) or RUN_TRAJECTORIES
    if need_baselines:
        baseline_spec = {
            "model": MODEL_KEY,
            "steps": run_steps,
            "max_new_tokens": run_steps + 1,
            "prompts": {
                row["id"]: prompts_by_id[row["id"]]
                for row in active_prompts
            },
        }
        if BASELINE_PATH.exists():
            baseline_payload = json.loads(BASELINE_PATH.read_text())
            if baseline_payload.get("spec") != baseline_spec:
                raise RuntimeError(
                    f"{BASELINE_PATH} does not match this run; use OVERWRITE=True"
                )
            baseline_texts_by_id = baseline_payload["texts"]
            print(f"reused {len(baseline_texts_by_id)} baselines")
        elif RUN_GENERATION:
            for prompt_row in active_prompts:
                prompt_id = prompt_row["id"]
                baseline_texts_by_id[prompt_id] = harness.generate_baseline(
                    [prompts_by_id[prompt_id]], max_new_tokens=run_steps + 1
                )[0]
            BASELINE_PATH.write_text(
                json.dumps(
                    {"spec": baseline_spec, "texts": baseline_texts_by_id},
                    indent=2,
                    ensure_ascii=False,
                )
                + "\n"
            )
            print(f"wrote {BASELINE_PATH}")
        else:
            raise FileNotFoundError(
                f"RUN_TRAJECTORIES requires an existing {BASELINE_PATH}"
            )
else:
    print("model load skipped")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-08-09 22:07:39.260 | INFO     | frames.models.hf.base:__init__:88 - Loaded model: meta-llama/Llama-3.1-8B-Instruct


2026-08-09 22:07:39.262 | WARNING  | frames.models.hf.base:__init__:89 - memory cost: 15316 Mb


loaded 18 cached concepts


  0%|                                                     | 0/1 [00:00<?, ?it/s]

...

  0%|                                                     | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.30s/it]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.31s/it]

wrote resources/53_long_composition_baselines.json


In [6]:
# Optional preflight benchmark. It is deliberately outside the main log.
if RUN_GENERATION and RUN_BENCHMARK:
    pair = active_pairs[0]
    prompt_row = active_prompts[0]
    ca, cb = concepts[pair["a"]], concepts[pair["b"]]
    guide = Concept.average([ca, cb])
    benchmark_rows = []
    for benchmark_steps in (24, 50, 100):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        text, _ = fur.generate_with_topk_multi_guide(
            [prompts_by_id[prompt_row["id"]]],
            guides=[guide],
            k=K,
            steps=benchmark_steps,
            fluency_weight=0.4,
            batch_size=1,
        )
        elapsed = time.perf_counter() - started
        continuation = harness.continuation(
            text[0], prompts_by_id[prompt_row["id"]]
        )
        meta = generation_metadata(
            fur.model.tokenizer, continuation, benchmark_steps, elapsed
        )
        benchmark_rows.append(
            {
                "steps": benchmark_steps,
                "elapsed_seconds": elapsed,
                "seconds_per_output_token": elapsed / max(meta["token_count"], 1),
                "continuation_tokens": meta["token_count"],
                "peak_cuda_gib": (
                    torch.cuda.max_memory_allocated() / 2**30
                    if torch.cuda.is_available()
                    else float("nan")
                ),
            }
        )
    benchmark = pd.DataFrame(benchmark_rows)
    benchmark.to_csv(TIMING_PATH, index=False)
    display(benchmark.round(3))


## Guided run

Each cell is generated and logged independently (`batch_size=1`). This is
conservative for the long context and makes interruption/resume exact. The
paper's greedy survivor rule is unchanged; only guide, score, or active
concept differs across the three methods.


In [7]:
FAMILY = {"blend": "frame", "vote": "score", "schedule": "time"}

def make_methods(prompt, pair, lam):
    ca, cb = concepts[pair["a"]], concepts[pair["b"]]
    blend_guide = Concept.average([ca, cb])
    return {
        "blend": lambda: fur.generate_with_topk_multi_guide(
            [prompt],
            guides=[blend_guide],
            k=K,
            steps=run_steps,
            fluency_weight=lam,
            batch_size=1,
        ),
        "vote": lambda: fur.generate_with_topk_multi_guide(
            [prompt],
            guides=[ca, cb],
            k=K,
            steps=run_steps,
            normalize="zscore",
            fluency_weight=lam,
            batch_size=1,
        ),
        "schedule": lambda: fur.generate_with_topk_multi_guide(
            [prompt],
            guides=[ca, cb],
            k=K,
            steps=run_steps,
            scorer=RoundRobin(),
            fluency_weight=lam,
            batch_size=1,
        ),
    }


if RUN_GENERATION and not BENCHMARK_ONLY:
    for pair in active_pairs:
        ca, cb = concepts[pair["a"]], concepts[pair["b"]]
        measured_rho = float(ca.rho(cb))
        for prompt_row in active_prompts:
            prompt_id = prompt_row["id"]
            prompt = prompts_by_id[prompt_id]
            for lam in active_lambdas:
                methods = make_methods(prompt, pair, lam)
                for method in METHODS:
                    key = (pair["id"], prompt_id, method, float(lam), run_steps)
                    if key in completed:
                        continue
                    torch.cuda.empty_cache()
                    started = time.perf_counter()
                    texts, probe = methods[method]()
                    elapsed = time.perf_counter() - started
                    continuation = harness.continuation(texts[0], prompt)
                    metadata = generation_metadata(
                        fur.model.tokenizer, continuation, run_steps, elapsed
                    )
                    harness.evaluate(
                        [prompt],
                        texts,
                        [pair["a"], pair["b"]],
                        config={
                            "experiment": "53",
                            "model": MODEL_KEY,
                            "pair_id": pair["id"],
                            "prompt_id": prompt_id,
                            "method": method,
                            "family": FAMILY[method],
                            "lam": float(lam),
                            "a": pair["a"],
                            "b": pair["b"],
                            "rho_catalog": float(pair["rho"]),
                            "rho_model": measured_rho,
                            "k": K,
                            "steps": run_steps,
                            "survivor_rule": "greedy",
                        },
                        probe=probe,
                        baseline_texts=[baseline_texts_by_id[prompt_id]],
                        concepts={pair["a"]: ca, pair["b"]: cb},
                        angles=True,
                        metadata=[metadata],
                    )
                    completed.add(key)
                    print(
                        f"{len(completed):>3}/{active_cells} "
                        f"{pair['id']} / {prompt_id} / {method} / "
                        f"lambda={lam}: {metadata['token_count']} tokens, "
                        f"{elapsed:.1f}s"
                    )


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:24<00:00, 24.54s/it]

100%|█████████████████████████████████████████████| 1/1 [00:24<00:00, 24.54s/it]

  1/450 education_sorrow / short_story / blend / lambda=0.0: 202 tokens, 24.5s


  0%|                                                     | 0/1 [00:00<?, ?it/s]

...

449/450 respite_peace / weather / vote / lambda=0.4: 205 tokens, 26.2s


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:26<00:00, 26.50s/it]

100%|█████████████████████████████████████████████| 1/1 [00:26<00:00, 26.50s/it]

450/450 respite_peace / weather / schedule / lambda=0.4: 207 tokens, 26.5s


## Whole-continuation summary

Summary rows retain pair identity and aggregate over prompts. Positive joint
values mean both concepts improved relative to the prompt-matched baseline.
Length fields use actual re-tokenized continuations, exposing the historical
one-seed-token offset and early EOS rather than assuming it away.


In [8]:
parsed = read_records(LOG_PATH)
result_rows = []
for record in parsed:
    config = record["config"]
    expression = record["expression"]
    angles = record["angles"]
    da = expression[config["a"]]["delta"]
    db = expression[config["b"]]["delta"]
    ca = angles[config["a"]]["delta"]["token_cos"]
    cb = angles[config["b"]]["delta"]["token_cos"]
    metadata = record.get("metadata", {})
    result_rows.append(
        {
            "pair_id": config["pair_id"],
            "prompt_id": config["prompt_id"],
            "a": config["a"],
            "b": config["b"],
            "rho": config["rho_model"],
            "method": config["method"],
            "lam": config["lam"],
            "joint_expression": min(da, db),
            "both_positive_expression": min(da, db) > 0,
            "joint_token_cos": min(ca, cb),
            "both_positive_token_cos": min(ca, cb) > 0,
            "ppl_ratio": record["ppl_ratio"],
            "continuation_tokens": metadata.get("token_count"),
            "termination": metadata.get("termination"),
            "distinct_token_ratio": metadata.get("distinct_token_ratio"),
            "repeated_4gram_rate": metadata.get("repeated_ngram_rate"),
            "max_repeated_span_tokens": metadata.get(
                "max_repeated_span_tokens"
            ),
            "elapsed_seconds": metadata.get("elapsed_seconds"),
        }
    )

results = pd.DataFrame(result_rows)
summary = pd.DataFrame()
if not results.empty:
    summary = (
        results.groupby(
            ["pair_id", "a", "b", "rho", "method", "lam"],
            dropna=False,
        )
        .agg(
            joint_expression=("joint_expression", "mean"),
            both_positive_expression=("both_positive_expression", "mean"),
            joint_token_cos=("joint_token_cos", "mean"),
            both_positive_token_cos=("both_positive_token_cos", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
            continuation_tokens=("continuation_tokens", "mean"),
            distinct_token_ratio=("distinct_token_ratio", "mean"),
            repeated_4gram_rate=("repeated_4gram_rate", "mean"),
            max_repeated_span_tokens=("max_repeated_span_tokens", "mean"),
            elapsed_seconds=("elapsed_seconds", "mean"),
            n_prompts=("prompt_id", "nunique"),
        )
        .reset_index()
    )
    summary["observed_records"] = len(results)
    summary["expected_records"] = active_cells
    summary["complete"] = len(results) == active_cells
    summary.to_csv(SUMMARY_PATH, index=False)
    display(
        results.groupby(["method", "lam"])
        .agg(
            joint_cos=("joint_token_cos", "mean"),
            both_positive=("both_positive_token_cos", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
            repeated_4gram=("repeated_4gram_rate", "mean"),
            tokens=("continuation_tokens", "mean"),
            n=("pair_id", "size"),
        )
        .round(4)
    )
    print(f"wrote {SUMMARY_PATH}; completeness {len(results)}/{active_cells}")
else:
    print(f"no records at {LOG_PATH}; generation remains disabled")


joint_cos  both_positive  ppl_ratio  repeated_4gram    tokens  \
method   lam                                                                  
blend    0.0     0.0151         0.8533     3.9514          0.3677  197.2800   
         0.4     0.0142         0.8267     2.6876          0.2954  189.6667   
schedule 0.0     0.0179         0.8933     3.7186          0.3458  203.5067   
         0.4     0.0128         0.8267     2.3347          0.2844  194.2000   
vote     0.0     0.0153         0.9467    10.5869          0.2541  123.5333   
         0.4     0.0106         0.8400     5.4540          0.1657  127.2667   

               n  
method   lam      
blend    0.0  75  
         0.4  75  
schedule 0.0  75  
         0.4  75  
vote     0.0  75  
         0.4  75

wrote resources/53_long_composition_summary.csv; completeness 450/450


## Optional 20-token trajectories

`RUN_TRAJECTORIES` performs one hidden-state pass per continuation and
baseline, then computes bin-level FRH expression and token-to-subspace cosine
from those hidden states. Baseline subtraction is reported only for bins with
tokens on both sides, with explicit coverage. Segment PPL is also recorded.
This is separate from generation so the expensive analysis can be resumed.


In [9]:
def token_hidden(text):
    tokens = fur.model.tokenize(text)
    with torch.inference_mode():
        hidden = fur.model.forward_last_hiden_state(tokens).squeeze(0)
    ids = tokens.squeeze(0).tolist()
    bos_id = fur.model.tokenizer.bos_token_id
    if ids and ids[0] == bos_id:
        ids = ids[1:]
        hidden = hidden[1:]
    return ids, hidden


def binned_values(ids, hidden, concept_map):
    bins = token_bins(ids, BIN_WIDTH)
    if not bins:
        return {}
    values = {part.index: {"bin": part} for part in bins}
    for name, concept in concept_map.items():
        projected = fur._project_hidden_states(
            hidden.unsqueeze(0), concept
        ).reshape(len(ids), -1).float().mean(-1)
        cosine = vector_subspace_cosine(
            hidden, concept.tensor.squeeze(0)
        ).float()
        for part in bins:
            values[part.index][name] = {
                "expression": float(projected[part.start : part.stop].mean()),
                "token_cos": float(cosine[part.start : part.stop].mean()),
            }
    return values


def segment_perplexity(token_ids):
    if len(token_ids) < 2:
        return float("nan")
    return harness.perplexity(fur.model.tokenizer.decode(token_ids))


trajectory_rows = []
if RUN_TRAJECTORIES:
    baseline_hidden = {}
    baseline_ppl = {}
    for record_index, record in enumerate(parsed, start=1):
        config = record["config"]
        prompt_id = config["prompt_id"]
        concept_map = {
            config["a"]: concepts[config["a"]],
            config["b"]: concepts[config["b"]],
        }
        steered_ids, steered_hidden = token_hidden(record["continuation"])
        steered = binned_values(steered_ids, steered_hidden, concept_map)
        if prompt_id not in baseline_hidden:
            baseline_continuation = harness.continuation(
                baseline_texts_by_id[prompt_id], prompts_by_id[prompt_id]
            )
            baseline_hidden[prompt_id] = token_hidden(baseline_continuation)
        base_ids, base_hidden = baseline_hidden[prompt_id]
        baseline = binned_values(base_ids, base_hidden, concept_map)

        for bin_index in sorted(set(steered) & set(baseline)):
            sbin = steered[bin_index]["bin"]
            bbin = baseline[bin_index]["bin"]
            bppl_key = (prompt_id, bin_index)
            if bppl_key not in baseline_ppl:
                baseline_ppl[bppl_key] = segment_perplexity(bbin.token_ids)
            segment_ppl = segment_perplexity(sbin.token_ids)
            row = {
                "pair_id": config["pair_id"],
                "prompt_id": prompt_id,
                "method": config["method"],
                "lam": config["lam"],
                "bin": bin_index,
                "start_token": sbin.start,
                "stop_token": sbin.stop,
                "coverage": min(len(sbin.token_ids), len(bbin.token_ids))
                / BIN_WIDTH,
                "segment_ppl": segment_ppl,
                "baseline_segment_ppl": baseline_ppl[bppl_key],
                "segment_ppl_ratio": segment_ppl / baseline_ppl[bppl_key],
            }
            for label, name in (("a", config["a"]), ("b", config["b"])):
                row[f"{label}_expression_steered"] = steered[bin_index][name][
                    "expression"
                ]
                row[f"{label}_expression_baseline"] = baseline[bin_index][name][
                    "expression"
                ]
                row[f"{label}_expression_delta"] = (
                    row[f"{label}_expression_steered"]
                    - row[f"{label}_expression_baseline"]
                )
                row[f"{label}_token_cos_steered"] = steered[bin_index][name][
                    "token_cos"
                ]
                row[f"{label}_token_cos_baseline"] = baseline[bin_index][name][
                    "token_cos"
                ]
                row[f"{label}_token_cos_delta"] = (
                    row[f"{label}_token_cos_steered"]
                    - row[f"{label}_token_cos_baseline"]
                )
            row["joint_expression_delta"] = min(
                row["a_expression_delta"], row["b_expression_delta"]
            )
            row["joint_token_cos_delta"] = min(
                row["a_token_cos_delta"], row["b_token_cos_delta"]
            )
            if config["method"] == "schedule":
                scored_positions = range(max(1, sbin.start), sbin.stop)
                active = [(position - 1) % 2 for position in scored_positions]
                row["schedule_a_fraction"] = (
                    active.count(0) / len(active) if active else float("nan")
                )
            else:
                row["schedule_a_fraction"] = float("nan")
            trajectory_rows.append(row)
        del steered_hidden
        if record_index % 10 == 0:
            torch.cuda.empty_cache()
            print(f"trajectory {record_index}/{len(parsed)}")

    trajectories = pd.DataFrame(trajectory_rows)
    trajectories.to_csv(TRAJECTORY_PATH, index=False)
    print(f"wrote {len(trajectories)} rows to {TRAJECTORY_PATH}")
    if not trajectories.empty:
        trajectory_summary = (
            trajectories.groupby(["method", "lam", "bin"])
            .agg(
                joint_token_cos=("joint_token_cos_delta", "mean"),
                joint_expression=("joint_expression_delta", "mean"),
                segment_ppl_ratio=("segment_ppl_ratio", "mean"),
                coverage=("coverage", "mean"),
            )
            .reset_index()
        )
        fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
        for axis, metric, ylabel in zip(
            axes,
            ("joint_token_cos", "joint_expression", "segment_ppl_ratio"),
            (
                "joint token-cosine delta",
                "joint expression delta",
                "segment PPL ratio",
            ),
        ):
            for (method, lam), cell in trajectory_summary.groupby(
                ["method", "lam"]
            ):
                axis.plot(
                    cell["bin"] * BIN_WIDTH,
                    cell[metric],
                    marker="o",
                    label=f"{method}, lambda={lam}",
                )
            axis.set_xlabel("continuation token position")
            axis.set_ylabel(ylabel)
        axes[0].axhline(0, color="gray", linewidth=0.8)
        axes[1].axhline(0, color="gray", linewidth=0.8)
        axes[0].legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(TRAJECTORY_FIGURE_PATH, dpi=150)
        plt.show()

        final_bins = (
            trajectories.sort_values("bin")
            .groupby(["pair_id", "prompt_id", "method", "lam"])
            .tail(1)
            .groupby(["pair_id", "method", "lam"])
            .agg(
                final_joint_expression=("joint_expression_delta", "mean"),
                final_joint_token_cos=("joint_token_cos_delta", "mean"),
                final_segment_ppl_ratio=("segment_ppl_ratio", "mean"),
                final_bin_coverage=("coverage", "mean"),
            )
            .reset_index()
        )
        summary = summary.merge(
            final_bins, on=["pair_id", "method", "lam"], how="left"
        )
        summary.to_csv(SUMMARY_PATH, index=False)


trajectory 10/450


trajectory 20/450


trajectory 30/450


trajectory 40/450


trajectory 50/450


...

trajectory 410/450


trajectory 420/450


trajectory 430/450


trajectory 440/450


trajectory 450/450
wrote 3101 rows to resources/53_long_composition_trajectories.csv


In [10]:
if not results.empty:
    plot_summary = (
        results.groupby(["method", "lam"])
        .agg(
            joint_token_cos=("joint_token_cos", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
            repeated_4gram_rate=("repeated_4gram_rate", "mean"),
        )
        .reset_index()
    )
    methods = list(METHODS)
    colors = {"blend": "tab:blue", "vote": "tab:orange", "schedule": "tab:green"}
    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
    for axis, metric, ylabel in zip(
        axes,
        ("joint_token_cos", "ppl_ratio", "repeated_4gram_rate"),
        ("joint token-cosine delta", "continuation PPL ratio", "repeated 4-gram rate"),
    ):
        for method in methods:
            cell = plot_summary[plot_summary["method"] == method].sort_values(
                "lam"
            )
            axis.plot(
                cell["lam"],
                cell[metric],
                marker="o",
                label=method,
                color=colors[method],
            )
        axis.set_xlabel("fluency weight lambda")
        axis.set_ylabel(ylabel)
    axes[0].axhline(0, color="gray", linewidth=0.8)
    axes[1].axhline(2.5, color="gray", linestyle=":", linewidth=0.8)
    axes[0].legend()
    plt.tight_layout()
    plt.savefig(FIGURE_PATH, dpi=150)
    plt.show()

    qualitative = []
    for record in parsed:
        config = record["config"]
        qualitative.append(
            {
                "pair": config["pair_id"],
                "prompt": config["prompt_id"],
                "method": config["method"],
                "lambda": config["lam"],
                "continuation": record["continuation"],
            }
        )
    display(pd.DataFrame(qualitative))


,pair,prompt,method,lambda,continuation
0,education_sorrow,short_story,blend,0.0,"Once, on one storm-ravined coast there existed..."
1,education_sorrow,short_story,vote,0.0,"Once, in ancient Greece on Mount Ida, there ex..."
2,education_sorrow,short_story,schedule,0.0,"Once, on one storm-raged night in late October..."
3,education_sorrow,short_story,blend,0.4,"Once, in a small village surrounded by vast de..."
4,education_sorrow,short_story,vote,0.4,"Once, in a small coastal town surrounded only ..."
...,...,...,...,...,...
445,respite_peace,weather,vote,0.0,Today the weather is not something I can obser...
446,respite_peace,weather,schedule,0.0,Today the weather is not something I can direc...
447,respite_peace,weather,blend,0.4,Today the weather is not something I can direc...
448,respite_peace,weather,vote,0.4,Today the weather is not something I can obser...
